<a href="https://www.kaggle.com/code/muhammaddhiyaulatha/arc-baseline-zero-model-ipynb?scriptVersionId=312713200" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# ARC Prize 2026 - Baseline Model

This notebook contains my first submission to the ARC-AGI-2 competition on Kaggle.

## Approach

* Generate output grids filled with zeros
* Match input grid dimensions
* Ensure correct submission format

## Purpose

This is a baseline to understand:

* Submission pipeline
* Evaluation system
* Dataset structure

Next step: implement rule-based reasoning.


In [1]:
import json
import os
import numpy as np
from collections import Counter

is_rerun = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))

# LOAD DATA
if is_rerun:
    path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
else:
    path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

with open(path) as f:
    data = json.load(f)

# BASIC UTILS
def copy_grid(g):
    return [row[:] for row in g]

def zero_grid(h, w):
    return [[0]*w for _ in range(h)]

def similarity(a, b):
    if len(a) != len(b) or len(a[0]) != len(b[0]):
        return 0
    same = sum(a[i][j] == b[i][j] for i in range(len(a)) for j in range(len(a[0])))
    return same / (len(a)*len(a[0]))

def most_common_color(grid):
    flat = [c for row in grid for c in row]
    return Counter(flat).most_common(1)[0][0]

# TRANSFORMATIONS
def rotate90(g):
    return list(map(list, zip(*g[::-1])))

def flip_h(g):
    return [row[::-1] for row in g]

def flip_v(g):
    return g[::-1]

# SCALING & TILING
def apply_scaling(inp, factor):
    arr = np.array(inp)
    return np.repeat(np.repeat(arr, factor, axis=0), factor, axis=1).tolist()

def apply_tiling(inp, target_h, target_w):
    arr = np.array(inp)
    reps = (target_h // len(inp), target_w // len(inp[0]))
    return np.tile(arr, reps).tolist()

def predict_output_shape(task, inp):
    h, w = len(inp), len(inp[0])
    for pair in task["train"]:
        in_h, in_w = len(pair["input"]), len(pair["input"][0])
        out_h, out_w = len(pair["output"]), len(pair["output"][0])

        if out_h % in_h == 0 and out_w % in_w == 0:
            return out_h, out_w
    return h, w

# SYMMETRY
def symmetry_complete(g):
    return [row + row[::-1] for row in g]

# GENERATE CANDIDATES
def generate_candidates(inp, task):
    h, w = len(inp), len(inp[0])
    target_h, target_w = predict_output_shape(task, inp)

    cands = []

    # basic
    cands.append(copy_grid(inp))
    cands.append(zero_grid(h, w))
    cands.append([[most_common_color(inp)]*w for _ in range(h)])

    # rotate/flip
    cands.append(rotate90(inp))
    cands.append(flip_h(inp))
    cands.append(flip_v(inp))

    # scaling
    if target_h > h:
        factor = target_h // h
        cands.append(apply_scaling(inp, factor))

    # tiling
    if target_h > h:
        cands.append(apply_tiling(inp, target_h, target_w))

    # symmetry
    cands.append(symmetry_complete(inp))

    return cands

# RULE VALIDATION (KEY UPGRADE)
def validate_rule(rule_func, task):
    for pair in task["train"]:
        pred = rule_func(pair["input"])
        if pred != pair["output"]:
            return False
    return True

# SOLVER
def solve_task(task):
    results = []

    for test_case in task["test"]:
        inp = test_case["input"]

        cands = generate_candidates(inp, task)

        # Try perfect rule first
        for c in cands:
            def rule(x, c=c):
                return c
            if validate_rule(lambda x: c, task):
                best1 = c
                best2 = c
                break
        else:
            # fallback similarity
            target = task["train"][-1]["output"] if task["train"] else None

            scored = []
            for c in cands:
                score = similarity(c, target) if target else 0
                scored.append((score, c))

            scored.sort(reverse=True, key=lambda x: x[0])

            best1 = scored[0][1]
            best2 = scored[1][1] if len(scored) > 1 else best1

        results.append({
            "attempt_1": best1,
            "attempt_2": best2
        })

    return results

# BUILD SUBMISSION
submission = {}

for task_id, task in data.items():
    submission[task_id] = solve_task(task)

# SAVE
with open("submission.json", "w") as f:
    json.dump(submission, f)

print("V24 FINAL SUBMISSION READY!")

V24 FINAL SUBMISSION READY!
